# Лабораторная работа 2 · Вариант 17

Попов Лев Алексеевич, группа ИКТб-3301-04-00

## Как априорные вероятности влияют на распознавание спама?

Сравним оценивание долей классов по обучающим данным с равными вероятностями ham и spam. Разбиение, словарь и сглаживание одинаковы.

## Шаг 0. Подготовка (как в части 1)

Папка `datasets` должна лежать рядом с блокнотом. В Google Colab загрузите датасет так же, как в части 1 — раскомментируйте нужный вариант в ячейке ниже (локально ничего делать не нужно).

In [1]:
# --- Google Colab, вариант А: архив лежит в корне Google Диска ---
# from google.colab import drive
# drive.mount('/content/drive')
# !apt-get install -y p7zip-full > /dev/null
# !7z x -p1 -o/content/ "/content/drive/MyDrive/datasets.zip" -y

# --- Google Colab, вариант Б: загрузка архива в текущую сессию ---
# from google.colab import files
# uploaded = files.upload()          # выберите datasets.zip
# !apt-get install -y p7zip-full > /dev/null
# !7z x -p1 datasets.zip -y

In [2]:
import os
import email
from email import policy
import pandas as pd

DATA_DIR = 'datasets/trec07delay/data/'
LABELS_FILE = 'datasets/trec07delay/delay/index'

assert os.path.isdir(DATA_DIR) and os.path.isfile(LABELS_FILE), \
    'Датасет не найден — распакуйте datasets.zip рядом с блокнотом (шаг 0)'

from pathlib import Path
from zipfile import ZipFile
from io import BytesIO

# Проверяем полноту распаковки до разбора почты.
# Если часть писем отсутствует, читаем их как данные из архива курса.
archive = None
def open_letter(path):
    if os.path.isfile(path):
        return open(path, 'rb')
    return BytesIO(archive.read(path, pwd=b'1'))

def text_email(path):
    """Извлекает тему и текстовое (text/plain) тело письма — та же функция, что в части 1."""
    try:
        with open_letter(path) as f:
            msg = email.message_from_binary_file(f, policy=policy.default)
        subject = str(msg.get('Subject') or '')
        parts = []
        for part in msg.walk():
            if part.get_content_type() == 'text/plain':
                try:
                    content = part.get_content()
                    if isinstance(content, str):
                        parts.append(content)
                except Exception:
                    pass
        return subject + ' ' + ' '.join(parts)
    except Exception:
        return ''

# Читаем метки: имя файла → 'spam'/'ham'
labels = {}
with open(LABELS_FILE) as f:
    for line in f:
        parts = line.split()
        if len(parts) == 2:
            labels[parts[1].split('/')[-1]] = parts[0]

print('Меток прочитано:', len(labels))
missing = [name for name in labels if not Path(DATA_DIR, name).is_file()]
if missing:
    archive = ZipFile('datasets.zip')
    entries = set(archive.namelist())
    assert all(DATA_DIR + name in entries for name in missing), 'Архив неполный'
print('Писем по разметке:', len(labels), '| из архива:', len(missing))

Меток прочитано: 10000
Писем по разметке: 10000 | из архива: 7


In [3]:
# Читаем все письма в список текстов X и список меток y (займёт до минуты)
X, y = [], []
empty = 0
filelist = sorted(labels, key=lambda name: int(name.split('.')[1]))
for filename in filelist:
    text = text_email(DATA_DIR + filename)
    if len(text.split()) <= 2:      # отсев пустых и HTML-only писем по сырому тексту;
        empty += 1                  # в части 1 он строже (после удаления стоп-слов), поэтому числа отличаются
        continue
    X.append(text)
    y.append(labels[filename])

print(f'Прочитано писем: {len(X)}, пропущено пустых: {empty}')
print('Пример текста (первые 200 символов):')
print(X[0][:200])

Прочитано писем: 9868, пропущено пустых: 132
Пример текста (первые 200 символов):
Generic Cialis, branded quality@  


In [4]:
from sklearn.feature_extraction.text import CountVectorizer

In [5]:
# Случайное разбиение 70/30; stratify=y сохраняет долю спама одинаковой в обеих частях
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=2, stratify=y)

print(f'train: {len(X_train)} писем, test: {len(X_test)} писем')
print('Доля спама: train {:.1%}, test {:.1%}'.format(
    sum(v == 'spam' for v in y_train) / len(y_train),
    sum(v == 'spam' for v in y_test) / len(y_test)))

train: 6907 писем, test: 2961 писем
Доля спама: train 80.3%, test 80.3%


In [6]:
# Строим словарь ПО ТРЕНИРОВОЧНОЙ ВЫБОРКЕ и векторизуем обе части
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)   # fit: составить словарь + посчитать слова
X_test_counts = vectorizer.transform(X_test)         # только посчитать — словарь уже зафиксирован

print('Размер словаря:', len(vectorizer.vocabulary_))
print('Матрица train:', X_train_counts.shape, '| ненулевых элементов:', X_train_counts.nnz)
print('Матрица test: ', X_test_counts.shape)

Размер словаря: 45547
Матрица train: (6907, 45547) | ненулевых элементов: 462165
Матрица test:  (2961, 45547)


In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import numpy as np

rows = []
for label, fit_prior in [('Базовая модель', True), ('Равные вероятности', False)]:
    model = MultinomialNB(fit_prior=fit_prior).fit(X_train_counts, y_train)
    pred = model.predict(X_test_counts)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=['ham', 'spam']).ravel()
    rows.append([label, np.exp(model.class_log_prior_)[list(model.classes_).index('spam')],
                 accuracy_score(y_test, pred), precision_score(y_test, pred, pos_label='spam'),
                 recall_score(y_test, pred, pos_label='spam'), fp, fn])
table = pd.DataFrame(rows, columns=['Модель', 'P(spam)', 'Accuracy', 'Precision spam', 'Recall spam', 'FP', 'FN'])
table.set_index('Модель').round(4)

,P(spam),Accuracy,Precision spam,Recall spam,FP,FN
Модель,,,,,,
Базовая модель,0.8028,0.9791,0.9936,0.9802,15,47
Равные вероятности,0.5000,0.9703,0.9944,0.9684,13,75


## Вывод

В обучающей выборке доля спама равна 80.28%. При равных априорных вероятностях короткому или неоднозначному письму нужны более весомые текстовые признаки, чтобы попасть в спам. Число ложных срабатываний изменилось с 15 до 13, а пропусков спама — с 47 до 75. Полнота обнаружения спама снизилась (98.02% → 96.84%), общая доля правильных ответов снизилась (97.91% → 97.03%).